# torch-cnmf Inference Tutorial

This notebook demonstrates how to run torch-cnmf on the PBMC3k dataset from scanpy.

Steps:
1. Load data
2. Prepare (normalize, select high-variance genes)
3. Factorize (run NMF)
4. Combine replicates
5. Select K
6. Consensus
7. Save results to MuData

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import anndata
import muon
import os
import torch

from torch_cnmf import cNMF

In [ ]:
# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Data

In [ ]:
adata = sc.datasets.pbmc3k()
adata

In [ ]:
# Save counts to h5ad for cNMF input
counts_fn = "example_output/pbmc3k.h5ad"
os.makedirs("example_output", exist_ok=True)
adata.write(counts_fn)

## 2. Set Parameters

In [ ]:
# NMF parameters
n_iter = 10
num_highvar_genes = 2000
K = [5, 7, 10]
seed = 14
beta_loss = "frobenius"
init = "random"
algo = "halsvar"
mode = "batch"
tol = 1e-7
use_gpu = True

# Batch mode parameters
batch_max_epoch = 500
batch_hals_tol = 0.005
batch_hals_max_iter = 1000

# Minibatch mode parameters
minibatch_max_epoch = 200
minibatch_size = 5000
minibatch_max_iter = 1000
minibatch_usage_tol = 0.005
minibatch_spectra_tol = 0.005
minibatch_shuffle = True

# Refit parameters
sk_cd_refit = True

# Consensus parameters
density_thresholds = [2.0]

# Paths
output_directory = "example_output/cNMF"
run_name = "pbmc_test"

## 3. Prepare

In [ ]:
cnmf_obj = cNMF(output_dir=output_directory, name=run_name)

In [ ]:
cnmf_obj.prepare(
    counts_fn=counts_fn,
    components=K,
    n_iter=n_iter,
    num_highvar_genes=num_highvar_genes,
    seed=seed,
    beta_loss=beta_loss,
    init=init,
    algo=algo,
    mode=mode,
    tol=tol,
    use_gpu=use_gpu,
    alpha_usage=0.0,
    alpha_spectra=0.0,
    l1_ratio_usage=0.0,
    l1_ratio_spectra=0.0,
    fp_precision="float",
    batch_max_epoch=batch_max_epoch,
    batch_hals_tol=batch_hals_tol,
    batch_hals_max_iter=batch_hals_max_iter,
    minibatch_max_epoch=minibatch_max_epoch,
    minibatch_size=minibatch_size,
    minibatch_max_iter=minibatch_max_iter,
    minibatch_usage_tol=minibatch_usage_tol,
    minibatch_spectra_tol=minibatch_spectra_tol,
    minibatch_shuffle=minibatch_shuffle,
    sk_cd_refit=sk_cd_refit,
)

## 4. Factorize

In [ ]:
cnmf_obj.factorize(total_workers=1, skip_completed_runs=True)

## 5. Combine

In [ ]:
cnmf_obj.combine()

## 6. K Selection

In [ ]:
cnmf_obj.k_selection_plot()

## 7. Consensus

In [ ]:
for k in K:
    for thresh in density_thresholds:
        cnmf_obj.consensus(k=k, density_threshold=thresh, show_clustering=True)

## 8. Load and Visualize Results

In [ ]:
selected_K = 7
selected_thresh = 2.0
thresh_str = str(selected_thresh).replace(".", "_")

usage_norm, gep_scores, gep_tpm, topgenes = cnmf_obj.load_results(
    K=selected_K, density_threshold=selected_thresh
)
usage_norm.columns = [f"Usage_{i}" for i in usage_norm.columns]
usage_norm.head()

In [ ]:
topgenes.head(20)

## 9. UMAP Visualization

In [ ]:
adata = sc.read(counts_fn)

# Get high-variance genes used by cNMF
hvgs_file = os.path.join(output_directory, run_name, f"{run_name}.overdispersed_genes.txt")
hvgs = open(hvgs_file).read().split("\n")
hvgs = [g for g in hvgs if g]  # remove empty strings

# Normalize and process
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
adata.raw = sc.pp.log1p(adata.copy(), copy=True)
adata = adata[:, hvgs]
sc.pp.scale(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=15)
sc.tl.umap(adata)

In [ ]:
adata.obs = pd.merge(left=adata.obs, right=usage_norm, how="left", left_index=True, right_index=True)
sc.pl.umap(adata, color=usage_norm.columns, use_raw=True, ncols=3, vmin=0, vmax=1)

## 10. Save Results to MuData

In [ ]:
for thresh in density_thresholds:
    thresh_str = str(thresh).replace(".", "_")
    for k in K:
        # Load consensus usages and spectra
        scores = pd.read_csv(
            f"{output_directory}/{run_name}/{run_name}.usages.k_{k}.dt_{thresh_str}.consensus.txt",
            sep="\t", index_col=0,
        )
        loadings = pd.read_csv(
            f"{output_directory}/{run_name}/{run_name}.spectra.k_{k}.dt_{thresh_str}.consensus.txt",
            sep="\t", index_col=0,
        )

        # Save as plain text
        loading_dir = f"{output_directory}/{run_name}/loading"
        os.makedirs(loading_dir, exist_ok=True)
        scores.to_csv(f"{loading_dir}/cNMF_scores_{k}_{thresh}.txt", sep="\t")
        loadings.T.to_csv(f"{loading_dir}/cNMF_loadings_{k}_{thresh}.txt", sep="\t")

        # Build AnnData for programs
        adata_tpm = sc.read(
            f"{output_directory}/{run_name}/cnmf_tmp/{run_name}.tpm.h5ad"
        )
        adata_tpm.var_names_make_unique()
        adata_tpm.obs_names_make_unique()

        prog_data = anndata.AnnData(X=scores.values, obs=adata_tpm.obs)
        prog_data.varm["loadings"] = loadings.values
        prog_data.uns["var_names"] = loadings.columns.values

        prog_dir = f"{output_directory}/{run_name}/prog_data"
        os.makedirs(prog_dir, exist_ok=True)
        prog_data.write(f"{prog_dir}/NMF_{k}_{thresh_str}.h5ad")

        # Build MuData
        mdata = muon.MuData({"rna": adata_tpm, "cNMF": prog_data})
        adata_dir = f"{output_directory}/{run_name}/adata"
        os.makedirs(adata_dir, exist_ok=True)
        mdata.write(f"{adata_dir}/cNMF_{k}_{thresh_str}.h5mu")

        print(f"Saved k={k}, threshold={thresh}")